In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

<font size="6" color="red"><b>ch14. 웹데이터수집1_정적_공공api</b></font>

# 1절. BeautifulSoup 과 parser
```pip install bs4 - 아나콘다에 설치되어있음```
- 공식사이트 : https://www.crummy.com/software/BeautifulSoup/
- Docs : https://www.crummy.com/software/BeautifulSoup/bs4/doc/

In [2]:
import requests # http 요청 처리 lib
from requests_file import FileAdapter

In [16]:
s = requests.Session() # http 요청 관리를 위한 세션 객체
s.mount("file://", FileAdapter())
response = s.get("file:///ai_x/source/01_python/data/ch14_sample.html")
response

<Response [200]>

In [4]:
response.status_code
# 200 : 정상 / 404 : 없는 페이지 / # 406 : get, post 오류

200

In [6]:
ex = response.content  # 바이너리 형식의 내용, 이미지 등을 분석하는데 필요
ex

b'<!DOCTYPE html>\r\n<html lang="en">\r\n<head>\r\n  <meta charset="UTF-8">\r\n</head>\r\n<body>\r\n  <h1 class="greeting css" id="text">Hello, CSS</h1>\r\n  <h1 class="css">Hi, CSS</h1>\r\n  <div id="subject">subject \xec\x84\xa0\xed\x83\x9d\xec\x9e\x90 \xec\x95\x88\xec\x9d\x98 \xeb\x82\xb4\xec\x9a\xa9</div>\r\n  <p>CSS \xec\x84\xa0\xed\x83\x9d\xec\x9e\x90\xeb\x8a\x94 \xeb\x8b\xa4\xec\x96\x91\xed\x95\x9c \xea\xb3\xb3\xec\x97\x90\xec\x84\x9c \xed\x99\x9c\xec\x9a\xa9\xeb\x90\xa9\xeb\x8b\x88\xeb\x8b\xa4</p>\r\n  <div class="contents">\r\n    \xec\x84\xa0\xed\x83\x9d\xec\x9e\x90\xeb\xa5\xbc \xec\x96\xb4\xeb\x96\xbb\xea\xb2\x8c \xec\x9e\x91\xec\x84\xb1\xed\x95\x98\xeb\x8a\x90\xeb\x83\x90\xec\x97\x90 \xeb\x94\xb0\xeb\x9d\xbc\r\n    <span>\xeb\x8b\xa4\xeb\xa5\xb8<b>\xec\x9a\x94\xec\x86\x8c\xea\xb0\x80 \xeb\xb0\x98\xed\x99\x98</b></span>\xeb\x90\xa9\xeb\x8b\x88\xeb\x8b\xa4\r\n  </div>\r\n  <div>CSS \xec\x84\xa0\xed\x83\x9d\xec\x9e\x90\xeb\x8a\x94 \xeb\x8b\xa4\xec\x96\x91\xed\x95\x9c \xea\xb3\

In [7]:
ex.decode('utf-8')  # 읽을 수 있는 utf-8로 디코딩

'<!DOCTYPE html>\r\n<html lang="en">\r\n<head>\r\n  <meta charset="UTF-8">\r\n</head>\r\n<body>\r\n  <h1 class="greeting css" id="text">Hello, CSS</h1>\r\n  <h1 class="css">Hi, CSS</h1>\r\n  <div id="subject">subject 선택자 안의 내용</div>\r\n  <p>CSS 선택자는 다양한 곳에서 활용됩니다</p>\r\n  <div class="contents">\r\n    선택자를 어떻게 작성하느냐에 따라\r\n    <span>다른<b>요소가 반환</b></span>됩니다\r\n  </div>\r\n  <div>CSS 선택자는 다양한 곳에 <b>활용</b>됩니다</div>\r\n</body>\r\n</html>'

In [17]:
response.text  # 위와 같다. 텍스트만 크롤링이 필요하면 앞으로는 이걸로 내용을 읽어올 예정

'<!DOCTYPE html>\r\n<html lang="en">\r\n<head>\r\n  <meta charset="UTF-8">\r\n</head>\r\n<body>\r\n  <h1 class="greeting css" id="text" title="greeting">Hello, CSS</h1>\r\n  <h1 class="css">Hi, CSS</h1>\r\n  <div id="subject">subject 선택자 안의 내용</div>\r\n  <p>CSS 선택자는 다양한 곳에서 활용됩니다</p>\r\n  <div class="contents">\r\n    선택자를 어떻게 작성하느냐에 따라\r\n    <span>다른<b>요소가 반환</b></span>됩니다\r\n  </div>\r\n  <div>CSS 선택자는 다양한 곳에 <b>활용</b>됩니다</div>\r\n</body>\r\n</html>'

In [18]:
# html 파싱
from bs4 import BeautifulSoup
soup = BeautifulSoup(response.content,  # == response.text
                    "html.parser")  # html.parser를 이용하면 자동 디코딩이 되네
soup

<!DOCTYPE html>

<html lang="en">
<head>
<meta charset="utf-8"/>
</head>
<body>
<h1 class="greeting css" id="text" title="greeting">Hello, CSS</h1>
<h1 class="css">Hi, CSS</h1>
<div id="subject">subject 선택자 안의 내용</div>
<p>CSS 선택자는 다양한 곳에서 활용됩니다</p>
<div class="contents">
    선택자를 어떻게 작성하느냐에 따라
    <span>다른<b>요소가 반환</b></span>됩니다
  </div>
<div>CSS 선택자는 다양한 곳에 <b>활용</b>됩니다</div>
</body>
</html>

In [22]:
# soup.select_one('선택자') : 해당 선택자 처음 하나만 가져온다
el = soup.select_one('h1')
print('el : ', el)
print('el.text : ', el.text)
print('el.string : ', el.string)
print('el의 속성들 : ', el.attrs)
print('el의 title 속성 : ', el.attrs['title'])
print('el의 title 속성 : ', el.attrs.get('title'))  # 오류방지를 위해 get 함수 사용 권장
print('el의 이름 : ', el.name)  # 태그 이름을 가져옴

el :  <h1 class="greeting css" id="text" title="greeting">Hello, CSS</h1>
el.text :  Hello, CSS
el.string :  Hello, CSS
el의 속성들 :  {'class': ['greeting', 'css'], 'id': 'text', 'title': 'greeting'}
el의 title 속성 :  greeting
el의 title 속성 :  greeting
el의 이름 :  h1


In [30]:
# soup.select('선택자') : 해당 선택자 전체 요소를 리스트로 가져온다
el_list = soup.select('h1')
[el.text for el in el_list]
print('el_list : ', el_list)
print('el_list의 텍스트들 : ', [el.text for el in el_list])  # 웹크롤링 할때 리스트컴프리헨션 많이 사용한다
print('el_list의 속성들 : ', [el.attrs for el in el_list]) 
print('el_list의 class속성 : ', [el.attrs.get('class') for el in el_list]) 

el_list :  [<h1 class="greeting css" id="text" title="greeting">Hello, CSS</h1>, <h1 class="css">Hi, CSS</h1>]
el_list의 텍스트들 :  ['Hello, CSS', 'Hi, CSS']
el_list의 속성들 :  [{'class': ['greeting', 'css'], 'id': 'text', 'title': 'greeting'}, {'class': ['css']}]
el_list의 class속성 :  [['greeting', 'css'], ['css']]


In [32]:
# soup.select_one(선택자) vs find(태그, 속성)
print('select_one : ', soup.select_one('h1.css'))
print('find : ', soup.find('h1', {'class':'css'}))
print('find : ', soup.find('h1', class_='css'))
print()
print('select_one : ', soup.select_one('h1#text'))
print('find : ', soup.find('h1', {'id':'text'}))

select_one :  <h1 class="greeting css" id="text" title="greeting">Hello, CSS</h1>
find :  <h1 class="greeting css" id="text" title="greeting">Hello, CSS</h1>
find :  <h1 class="greeting css" id="text" title="greeting">Hello, CSS</h1>

select_one :  <h1 class="greeting css" id="text" title="greeting">Hello, CSS</h1>
find :  <h1 class="greeting css" id="text" title="greeting">Hello, CSS</h1>


In [36]:
# soup.select(선택자) vs find_all(태그, 속성)
print('모든 h1.css, span 태그(select) : ', 
      soup.select('h1.css, span'))
print('모든 h1.css, span 태그(find_all) : ', 
      soup.find_all(['h1', 'span'], {'class':'css'})) # span은 클래스가 없어서 선택되지 않는 문제가 있다

모든 h1.css, span 태그(select) :  [<h1 class="greeting css" id="text" title="greeting">Hello, CSS</h1>, <h1 class="css">Hi, CSS</h1>, <span>다른<b>요소가 반환</b></span>]
모든 h1.css, span 태그(find_all) :  [<h1 class="greeting css" id="text" title="greeting">Hello, CSS</h1>, <h1 class="css">Hi, CSS</h1>]


In [41]:
# 없는 엘리먼트 찾기 (4가지 방법)
print('find_all : ', soup.find_all('a', class_='css'))  # 빈리스트
print('find : ', soup.find('a', class_='css'))  # None
print('select : ', soup.select('a.css'))  # 빈리스트
print('select_one : ', soup.select_one('a.css'))  # None
# 여러 개를 찾는 함수는 빈리스트로, 한 개를 찾는 함수는 None으로 출력됨을 알 수 있다

find_all :  []
find :  None
select :  []
select_one :  None


# 2절. 정적 웹데이터 수집(정적 웹크롤링)
- json, html, xml
## 2.1 JSON파일
- requests 모듈(get)
- urllib.request모듈(urlopen)
- 실습페이지 - http://api.github.com

In [ ]:
# 크롤링 허용 범위는 사이트마다 다름 ~/robots.txt에서 확인할 수 있다

In [130]:
# 방법2
from urllib.request import urlopen
response = urlopen('http://api.github.com')
response

HTTPError: HTTP Error 403: rate limit exceeded

In [138]:
text = '{"속성1":"값1", "속성2":"값2", "속성3":"값3", "속성4":"값4"}'

In [139]:
# 문자(딕셔너리 타입)를 딕셔너리로
# "{'속성1':'값1', '속성2':'값2', '속성3':'값3', '속성4':'값4'}" => {'속성1':'값1', '속성2':'값2', '속성3':'값3', '속성4':'값4'}
import json
json.loads(text)

{'속성1': '값1', '속성2': '값2', '속성3': '값3', '속성4': '값4'}

## 2.2 html 크롤링
- https://finance.naver.com/marketindex/
- 크롤링 가능범위 확인 : https://finance.naver.com/marketindex/robots.txt
### 1) 네이버증권 시장지표 크롤링 실습
- https://finance.naver.com/marketindex/

In [70]:
# 방법 1 requests.get
import requests
from bs4 import BeautifulSoup
url = 'https://finance.naver.com/marketindex'
response = requests.get(url)
# response, response.status_code
# response.content.decode('cp949') == response.text
soup = BeautifulSoup(response.text, 'html.parser')
soup


<script language="javascript" src="/template/head_js.naver?referer=info.finance.naver.com&amp;menu=marketindex&amp;submenu=market"></script>
<script src="https://ssl.pstatic.net/imgstock/static.pc/20250522145100/js/info/jindo.min.ns.1.5.3.euckr.js" type="text/javascript"></script>
<script src="https://ssl.pstatic.net/imgstock/static.pc/20250522145100/js/jindo.1.5.3.element-text-patch.js" type="text/javascript"></script>
<div id="container" style="padding-bottom:0px;">
<div class="market_include">
<div class="market_data">
<div class="market1">
<div class="title">
<h2 class="h_market1"><span>환전 고시 환율</span></h2>
</div>
<!-- data -->
<div class="data">
<ul class="data_lst" id="exchangeList">
<li class="on">
<a class="head usd" href="/marketindex/exchangeDetail.naver?marketindexCd=FX_USDKRW" onclick="clickcr(this, 'fr1.usdt', '', '', event);">
<h3 class="h_lst"><span class="blind">미국 USD</span></h3>
<div class="head_info point_dn">
<span class="value">1,374.70</span>
<span class="txt_krw

In [69]:
# 방법 2 urlopen
from urllib.request import urlopen
response = urlopen(url)
# response, response.status # 상태코드
# response.read()  # requests의 content 와 동일
# response.read().decode('cp949')  # requests의 text 와 동일
soup = BeautifulSoup(response, 'html.parser')
soup


<script language="javascript" src="/template/head_js.naver?referer=info.finance.naver.com&amp;menu=marketindex&amp;submenu=market"></script>
<script src="https://ssl.pstatic.net/imgstock/static.pc/20250522145100/js/info/jindo.min.ns.1.5.3.euckr.js" type="text/javascript"></script>
<script src="https://ssl.pstatic.net/imgstock/static.pc/20250522145100/js/jindo.1.5.3.element-text-patch.js" type="text/javascript"></script>
<div id="container" style="padding-bottom:0px;">
<div class="market_include">
<div class="market_data">
<div class="market1">
<div class="title">
<h2 class="h_market1"><span>환전 고시 환율</span></h2>
</div>
<!-- data -->
<div class="data">
<ul class="data_lst" id="exchangeList">
<li class="on">
<a class="head usd" href="/marketindex/exchangeDetail.naver?marketindexCd=FX_USDKRW" onclick="clickcr(this, 'fr1.usdt', '', '', event);">
<h3 class="h_lst"><span class="blind">미국 USD</span></h3>
<div class="head_info point_dn">
<span class="value">1,374.70</span>
<span class="txt_krw

In [103]:
# 분석
# title => h3.h_lst > span.blind
# price => div.head_info > span.value
# unit => div.head_info > span > span.blind
# status => div.head_info > span.blind
title = soup.select('h3.h_lst > span.blind')
title = [t.text for t in title]
price = soup.select('div.head_info > span.value')
price = [float(p.text.replace(',','')) for p in price]
unit = soup.select('div.head_info > span > span.blind')
unit = [u.text for u in unit]
unit.insert(7,'')  # 7번째인 달러인덱스에는 단위가 없어 빈값 삽입
status = soup.select('div.head_info > span.blind')
status = [s.text for s in status]
status

['하락', '하락', '하락', '하락', '상승', '하락', '하락', '상승', '하락', '하락', '하락', '상승']

In [96]:
for idx in range(len(title)):
    print('{} : {}{} - {}'.format(title[idx], 
                                  price[idx], 
                                  unit[idx], 
                                  status[idx]))

미국 USD : 1374.7원 - 하락
일본 JPY(100엔) : 952.57원 - 하락
유럽연합 EUR : 1555.89원 - 하락
중국 CNY : 190.99원 - 하락
달러/일본 엔 : 144.38엔 - 상승
유로/달러 : 1.1342달러 - 하락
영국 파운드/달러 : 1.352달러 - 하락
달러인덱스 : 99.42 - 상승
WTI : 60.89달러 - 하락
휘발유 : 1633.17원 - 하락
국제 금 : 3300.4달러 - 하락
국내 금 : 146218.18원 - 상승


In [108]:
for idx, (t, p, u, s) in enumerate(zip(title, price, unit, status)):
    print(f'{idx+1}. {t} : {p}{u} - {s}')

1. 미국 USD : 1374.7원 - 하락
2. 일본 JPY(100엔) : 952.57원 - 하락
3. 유럽연합 EUR : 1555.89원 - 하락
4. 중국 CNY : 190.99원 - 하락
5. 달러/일본 엔 : 144.38엔 - 상승
6. 유로/달러 : 1.1342달러 - 하락
7. 영국 파운드/달러 : 1.352달러 - 하락
8. 달러인덱스 : 99.42 - 상승
9. WTI : 60.89달러 - 하락
10. 휘발유 : 1633.17원 - 하락
11. 국제 금 : 3300.4달러 - 하락
12. 국내 금 : 146218.18원 - 상승


### 2) 동행복권 크롤링 실습
- https://dhlottery.co.kr/gameResult.do?method=byWin

In [145]:
# 방법 1
response = requests.get('https://dhlottery.co.kr/gameResult.do?method=byWin')
soup = BeautifulSoup(response.text, 'html.parser')
soup


<!DOCTYPE html>

<html lang="ko">
<head>
<meta charset="utf-8"/>
<meta content="동행복권" id="utitle" name="title"/>
<meta content="동행복권 1173회 당첨번호 1,5,18,20,30,35+3. 1등 총 24명, 1인당 당첨금액 1,179,946,063원." id="desc" name="description"/>
<title>로또6/45 - 회차별 당첨번호</title>
<title>동행복권</title>
<meta content="IE=edge" http-equiv="X-UA-Compatible"/>
<link href="/images/common/favicon.ico" rel="shortcut icon" type="image/x-icon"/>
<link href="/images/common/favicon.ico" rel="icon" type="image/x-icon"/>
<script src="/js/jquery-1.9.1.min.js" type="text/javascript"></script>
<script src="/js/jquery-ui.js" type="text/javascript"></script>
<script charset="utf-8" src="/js/common.js" type="text/javascript"></script>
<script type="text/javascript">

fn_g_init_message("");

var gameUserId = "";

function goGame(){
	var userId = "";
	
	if(userId == '' || userId == null){
		alert("로그인 후 사용 해주시기 바랍니다.");
		location.href = "/user.do?method=login";
		return;
	}
	
	$.ajax({
		type:"get",                          

In [163]:
# 사이트 분석 
# 회차 turn = div.win_result > h4 > strong
# 날짜 date = div.win_result > p.desc'
# 당첨번호 = div.win > strong
# nums = div.win > p > span / 
# 보너스번호 = div.bonus > strong
# bonus = div.bonus > p > span
turn = soup.select_one('div.win_result > h4 > strong')
date = soup.select_one('div.win_result > p.desc')
n_title = soup.select_one('div.num.win > strong')
nums = soup.select('div.num.win > p > span')
b_title = soup.select_one('div.num.bonus > strong')
bonus = soup.select_one('div.num.bonus > p > span')
num = [int(n.text) for n in nums]
print('{}{}'.format(turn.text, date.text))
print('{} {}'.format(n_title.text, num))
print('{} {}'.format(b_title.text, bonus.text))

1173회(2025년 05월 24일 추첨)
당첨번호 [1, 5, 18, 20, 30, 35]
보너스 3


In [164]:
# 방법 2
response = urlopen('https://dhlottery.co.kr/gameResult.do?method=byWin')
soup = BeautifulSoup(response, 'html.parser')
soup


<!DOCTYPE html>

<html lang="ko">
<head>
<meta charset="utf-8"/>
<meta content="동행복권" id="utitle" name="title"/>
<meta content="동행복권 1173회 당첨번호 1,5,18,20,30,35+3. 1등 총 24명, 1인당 당첨금액 1,179,946,063원." id="desc" name="description"/>
<title>로또6/45 - 회차별 당첨번호</title>
<title>동행복권</title>
<meta content="IE=edge" http-equiv="X-UA-Compatible"/>
<link href="/images/common/favicon.ico" rel="shortcut icon" type="image/x-icon"/>
<link href="/images/common/favicon.ico" rel="icon" type="image/x-icon"/>
<script src="/js/jquery-1.9.1.min.js" type="text/javascript"></script>
<script src="/js/jquery-ui.js" type="text/javascript"></script>
<script charset="utf-8" src="/js/common.js" type="text/javascript"></script>
<script type="text/javascript">

fn_g_init_message("");

var gameUserId = "";

function goGame(){
	var userId = "";
	
	if(userId == '' || userId == null){
		alert("로그인 후 사용 해주시기 바랍니다.");
		location.href = "/user.do?method=login";
		return;
	}
	
	$.ajax({
		type:"get",                          

In [184]:
# find, find_all 이용
# 사이트 분석 
# 회차 turn = div.win_result > h4 > strong
# 날짜 date = div.win_result > p.desc'
# 당첨번호 n_title = div.win > strong
# nums = div.win > p > span 
# 보너스번호 b_title = div.bonus > strong
# bonus = div.bonus > p > span
turn = soup.find('div', class_='win_result').find('h4').find('strong').text
date = soup.find('div', class_='win_result').find('p', class_='desc').text
n_title = soup.find('div', class_='win').find('strong').text
nums = soup.find('div', class_='win').find('p').find_all('span')
num = [int(n.text) for n in nums]
b_title = soup.find('div', class_='bonus').find('strong').text
bonus = int(soup.find('div', class_='bonus').find('p').find('span').text)
print(f'{turn} {date}')
print(f'{n_title} {num}')
print(f'{b_title} {bonus}')

1173회 (2025년 05월 24일 추첨)
당첨번호 [1, 5, 18, 20, 30, 35]
보너스 3


### 3) daum 검색 리스트(뉴스페이지)

In [191]:
# 방법 1-1. 딕셔너리 리스트
import requests
from bs4 import BeautifulSoup
keyword = '비트코인'
url = "https://search.daum.net/search?w=news&nil_search=btn&DA=NTB&enc=utf8&cluster=y&cluster_page=1&q="+keyword
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')
itmes_find_list = [] # 검색할 결과를 담을 딕셔너리 리스트

In [207]:
# 사이트 분석
# 제목 title = div.item-title > a
titles = soup.select('div.item-title > strong.tit-g.clamp-g > a')
itmes_find_list = [] 
for i, title in enumerate(titles):
    itmes_find_list.append({'no': i, 
                            'title' : title.text,
                            'link' : title.attrs['href']})

import pandas as pd    
pd.DataFrame(itmes_find_list)

,no,title,link
0,0,[비즈 나우] 비트코인 2025 컨퍼런스 개막…'전략자산' 선언 코앞,http://v.daum.net/v/20250528075215864
1,1,[비트코인 2025] 백악관 크립토 차르 “美 정부 비트코인 추가 매입 검토…부채...,http://v.daum.net/v/20250528103907230
2,2,"비트코인, 트럼프 미디어 비축 소식에도 주춤…1억5100만원대",http://v.daum.net/v/20250528095112897
3,3,“맥O날드보다 맛없어!” 혹평이 가득한 트럼프 만찬과 비트코인 피자데이[엠블록레터],http://v.daum.net/v/20250528143002735
4,4,"美 상원의원 ""트럼프 대통령, 비트코인법 지지""",http://v.daum.net/v/20250528090342853
5,5,[비트코인 2025] 로빈후드 창업자 “토큰화 증권은 美 ‘자본 패권’ 키우는 수단”,http://v.daum.net/v/20250528110600761
6,6,"'비트코인 빚투' 스트레티지, 또 샀다…보유량 58만개 돌파",http://v.daum.net/v/20250528042404292
7,7,"""비트코인 산다""…트럼프家, 25억달러 자금 조달 추진[코인브리핑]",http://v.daum.net/v/20250528111837527
8,8,"숨 고르는 비트코인, 10만8000달러선 '주춤'",http://v.daum.net/v/20250528095921258
9,9,"트럼프미디어그룹, 25억 달러 규모 자금 조달 통해 비트코인 매입 예고",http://v.daum.net/v/20250528084245020


In [220]:
# 방법 1-2. 2차원 리스트
import requests
from bs4 import BeautifulSoup
keyword = '비트코인'
url = "https://search.daum.net/search?w=news&nil_search=btn&DA=NTB&enc=utf8&cluster=y&cluster_page=1&q="+keyword+'p='+page_n
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')
itmes_find_list = [] # 검색할 결과를 담을 2차원 리스트
titles = soup.select('div.item-title > strong.tit-g.clamp-g > a')
for i, title in enumerate(titles):
    itmes_find_list.append([i, title.text, title.attrs['href']])

import pandas as pd    
pd.DataFrame(itmes_find_list, columns=['no', 'title', 'link'])

,no,title,link
0,0,[비트코인 2025] 백악관 크립토 차르 “美 정부 비트코인 추가 매입 검토…부채...,http://v.daum.net/v/20250528103907230
1,1,[비즈 나우] 비트코인 2025 컨퍼런스 개막…'전략자산' 선언 코앞,http://v.daum.net/v/20250528075215864
2,2,“맥O날드보다 맛없어!” 혹평이 가득한 트럼프 만찬과 비트코인 피자데이[엠블록레터],http://v.daum.net/v/20250528143002735
3,3,"블랙록, 자사 비트코인 ETF 보유량 25% 확대…기관 투자 본격화 신호탄",http://v.daum.net/v/20250528151802080
4,4,"비트코인, 트럼프 미디어 비축 소식에도 주춤…1억5100만원대",http://v.daum.net/v/20250528095112897
5,5,"'비트코인 빚투' 스트레티지, 또 샀다…보유량 58만개 돌파",http://v.daum.net/v/20250528042404292
6,6,"美 상원의원 ""트럼프 대통령, 비트코인법 지지""",http://v.daum.net/v/20250528090342853
7,7,"숨 고르는 비트코인, 10만8000달러선 '주춤'",http://v.daum.net/v/20250528095921258
8,8,[비트코인 2025] 로빈후드 창업자 “토큰화 증권은 美 ‘자본 패권’ 키우는 수단”,http://v.daum.net/v/20250528110600761
9,9,"트럼프미디어그룹, 25억 달러 규모 자금 조달 통해 비트코인 매입 예고",http://v.daum.net/v/20250528084245020


In [224]:
# 페이지를 이동하며 크롤링
# 다음 뉴스 검색(키워드, 원하는 페이지수)
import requests
from bs4 import BeautifulSoup
import pandas as pd   
import time
keyword = '비트코인'
page = 1
# url = f"https://search.daum.net/search?w=news&nil_search=btn&DA=NTB&enc=utf8&cluster=y&cluster_page=1&q={keyword}&p={page}"
url = f"https://search.daum.net/search?w=news&nil_search=btn&DA=NTB&enc=utf8" # 필요없을 것 같은거 지워버림
params = {'q':keyword, 'p':page}   # 파라미터가 많은 경우에 이와 같이 처리함
response = requests.get(url, params=params)
soup = BeautifulSoup(response.text, 'html.parser')
itmes_find_list = [] # 검색할 결과를 담을 2차원 리스트
titles = soup.select('div.item-title > strong.tit-g.clamp-g > a')
for i, title in enumerate(titles):
    itmes_find_list.append([(page-1)*10 + i, title.text, title.attrs['href']])

import pandas as pd    
pd.DataFrame(itmes_find_list, columns=['no', 'title', 'link'])

# 아래에 함수로 간편하게 선언하자

,no,title,link
0,0,[비트코인 2025] 백악관 크립토 차르 “美 정부 비트코인 추가 매입 검토…부채...,http://v.daum.net/v/20250528103907230
1,1,[비즈 나우] 비트코인 2025 컨퍼런스 개막…'전략자산' 선언 코앞,http://v.daum.net/v/20250528075215864
2,2,美·日 국채 팔고 비트코인 샀다…'안전자산' 등극,http://v.daum.net/v/20250528154126316
3,3,“맥O날드보다 맛없어!” 혹평이 가득한 트럼프 만찬과 비트코인 피자데이[엠블록레터],http://v.daum.net/v/20250528143002735
4,4,"블랙록, 자사 비트코인 ETF 보유량 25% 확대…기관 투자 본격화 신호탄",http://v.daum.net/v/20250528151802080
5,5,"'비트코인 빚투' 스트레티지, 또 샀다…보유량 58만개 돌파",http://v.daum.net/v/20250528042404292
6,6,"비트코인, 트럼프 미디어 비축 소식에도 주춤…1억5100만원대",http://v.daum.net/v/20250528095112897
7,7,"美 상원의원 ""트럼프 대통령, 비트코인법 지지""",http://v.daum.net/v/20250528090342853
8,8,"숨 고르는 비트코인, 10만8000달러선 '주춤'",http://v.daum.net/v/20250528095921258
9,9,[비트코인 2025] 로빈후드 창업자 “토큰화 증권은 美 ‘자본 패권’ 키우는 수단”,http://v.daum.net/v/20250528110600761


In [241]:
# 함수 다음 뉴스 검색(키워드, 원하는 페이지수)
import requests
from bs4 import BeautifulSoup
import pandas as pd   
import time

def daum_news_crawling(keyword, page=1):
    '''
    다음 뉴스탭에서 키워드로 다음 검색한 결과를 return
    '''    
    url = f"https://search.daum.net/search?w=news&nil_search=btn&DA=NTB&enc=utf8"
    params = {'q':keyword, 'p':page} 
    response = requests.get(url, params=params)
    soup = BeautifulSoup(response.text, 'html.parser')
    itmes_find_list = [] # 검색할 결과를 담을 2차원 리스트
    titles = soup.select('div.item-title > strong.tit-g.clamp-g > a')
    for i, title in enumerate(titles):
        itmes_find_list.append({'no': (page-1)*10 + i, 
                            'title' : title.text,
                            'link' : title.attrs['href']})
    return itmes_find_list

In [243]:
result = []
keyword = '비트코인'
pages = 3
for page in range(1,pages+1):
    result.extend( daum_news_crawling(keyword, page) )
    time.sleep(1)  # 1초씩 시간을 두고 수행
# len(result), result

result_df = pd.DataFrame(result)
result_df

,no,title,link
0,0,美·日 국채 팔고 비트코인 샀다…'안전자산' 등극,http://v.daum.net/v/20250528154126316
1,1,"블랙록, 자사 비트코인 ETF 보유량 25% 확대…기관 투자 본격화 신호탄",http://v.daum.net/v/20250528151802080
2,2,[비트코인 2025] 백악관 크립토 차르 “美 정부 비트코인 추가 매입 검토…부채...,http://v.daum.net/v/20250528103907230
3,3,[비즈 나우] 비트코인 2025 컨퍼런스 개막…'전략자산' 선언 코앞,http://v.daum.net/v/20250528075215864
4,4,“맥O날드보다 맛없어!” 혹평이 가득한 트럼프 만찬과 비트코인 피자데이[엠블록레터],http://v.daum.net/v/20250528143002735
5,5,"비트코인, 트럼프 미디어 비축 소식에도 주춤…1억5100만원대",http://v.daum.net/v/20250528095112897
6,6,"美 상원의원 ""트럼프 대통령, 비트코인법 지지""",http://v.daum.net/v/20250528090342853
7,7,[비트코인 2025] 로빈후드 창업자 “토큰화 증권은 美 ‘자본 패권’ 키우는 수단”,http://v.daum.net/v/20250528110600761
8,8,"트럼프미디어그룹, 25억 달러 규모 자금 조달 통해 비트코인 매입 예고",http://v.daum.net/v/20250528084245020
9,9,"'비트코인 빚투' 스트레티지, 또 샀다…보유량 58만개 돌파",http://v.daum.net/v/20250528042404292


In [246]:
keywords = ['청바지', '면바지']
pages = 5
result0 = [] # 청바지 검색결과 50개
result1 = [] # 면바지 검색결과 50개
for i, keyword in enumerate(keywords):
    for page in range(1, pages+1):
        print(f'~ ~ ~ {i} 번째 {keyword} {page}p 검색중입니다 ~ ~ ~')
        if i==0:
            result0.extend( daum_news_crawling(keyword, page) )
        else:
            result1.extend( daum_news_crawling(keyword, page) )
        time.sleep(3)
else:
    print('------작업이 완료되었습니다------')

~ ~ ~ 0 번째 청바지 1p 검색중입니다 ~ ~ ~
~ ~ ~ 0 번째 청바지 2p 검색중입니다 ~ ~ ~
~ ~ ~ 0 번째 청바지 3p 검색중입니다 ~ ~ ~
~ ~ ~ 0 번째 청바지 4p 검색중입니다 ~ ~ ~
~ ~ ~ 0 번째 청바지 5p 검색중입니다 ~ ~ ~
~ ~ ~ 1 번째 면바지 1p 검색중입니다 ~ ~ ~
~ ~ ~ 1 번째 면바지 2p 검색중입니다 ~ ~ ~
~ ~ ~ 1 번째 면바지 3p 검색중입니다 ~ ~ ~
~ ~ ~ 1 번째 면바지 4p 검색중입니다 ~ ~ ~
~ ~ ~ 1 번째 면바지 5p 검색중입니다 ~ ~ ~
------작업이 완료되었습니다------


In [247]:
result0_df = pd.DataFrame(result0)
result1_df = pd.DataFrame(result1)
display(result0_df,result1_df)

,no,title,link
0,0,"'이병헌♥' 이민정, 매장 통째로 빌려 청바지 쇼핑",http://v.daum.net/v/20250528050507783
1,1,"“추앙받을 만 하네”… 김지원, 셔츠에 청바지만 입어도 공항 ‘정지’",http://v.daum.net/v/20250526182411395
2,2,흰 티셔츠에 청바지 정석대로 입는 법,http://v.daum.net/v/20250527185425864
3,3,"이민정, 매장 빌려 폭풍 쇼핑 ""10년 전 샀던 청바지는 응급실""(MJ)",http://v.daum.net/v/20250527191015309
4,4,"‘이병헌♥’ 이민정, 청바지 사는데 매장 통째로 빌렸다..남다른 쇼핑 클래스 “나...",http://v.daum.net/v/20250527192310643
5,5,이시영처럼 다리 길어보이고 싶어서 청바지 분석함 [입스타그램],http://v.daum.net/v/20250519172231644
6,6,"'의사 그만둔' 고윤정, 흰 티+청바지...""정석 미녀"" [★해시태그]",http://v.daum.net/v/20250522182007490
7,7,"""청렴이 최고"" … 창원특례시 성산구 문화위생과, '청바지 데이' 추진",http://v.daum.net/v/20250511091428626
8,8,"패셔니스타 김지원, 청바지가 어울리는 여자랍니다! [포토]",http://v.daum.net/v/20250518220139191
9,9,"박병은 ""류준하로 3개월 활동""…청바지 모델 시절 언급",http://v.daum.net/v/20250519190323957


,no,title,link
0,0,사천에서 쾌청한 날씨 속 탁 트인 남해 풍경 즐겨요!,http://v.daum.net/v/20250528104429529
1,1,"윤승아, ♥김무열 쏙 빼닮은 깜찍 子 하객룩 공개‥벌써 우월한 유전자",http://v.daum.net/v/20250526145941690
2,2,"이번 주 포천 여행,반팔 하나로 충분할까?",http://v.daum.net/v/20250527171732518
3,3,"이재명, 첫 대학 방문…이준석 겨냥?",http://v.daum.net/v/20250526191942751
4,4,"""나는 젊어"" 팔소매 접은 이준석의 '흰셔츠' 속에도 뼈가 담겼다[21대 대선 리...",http://v.daum.net/v/20250525070013619
5,5,이재명 ‘간담회’ 이준석 ‘학식’…20대 표심 공략 [21대 대선],http://v.daum.net/v/20250526171503637
6,6,"대학생 만난 이재명 ""청년 극소수가 극우화…근묵자흑처럼 오염""",http://v.daum.net/v/20250526141950437
7,7,'배우자 토론' 자신감 얻은 김문수…딸·사위까지 총출동,http://v.daum.net/v/20250522185207535
8,8,정동원 첫사랑 품은 순정남 변신 ‘꽃등’ 라이브 클립 공개,http://v.daum.net/v/20250519123006455
9,9,"장년 품격을 높이는 '피어스 브로스넌'의 패션 꿀팁 3종 세트[長靑年, 늘 푸른 ...",http://v.daum.net/v/20250514043027212


In [252]:
result0_df.to_csv('data/ch14_'+keywords[0]+'.csv', index=False, encoding='cp949')
result1_df.to_csv('data/ch14_'+keywords[1]+'.csv', index=False)

### 4) user-agent를 추가하여 크롤링
- urlopen() 함수를 사용하면 크롤링이 안되는 사이트
- user-agent를 추가하여 크롤링

In [258]:
# 방법2
from urllib.request import urlopen, Request
# response = urlopen(url)  # url 오픈에서는 한글이 url에 들어가있으면 에러가 난다
keyword = '비트코인'
url = "https://search.daum.net/search?w=news&nil_search=btn&DA=NTB&enc=utf8&cluster=y&cluster_page=1&q="+keyword
import urllib.parse

keyword = urllib.parse.quote(keyword) # '비트코인' => '%EB%B9%84%ED%8A%B8%EC%BD%94%EC%9D%B8'
url = "https://search.daum.net/search?w=news&nil_search=btn&DA=NTB&enc=utf8&cluster=y&cluster_page=1&q="+keyword
headers = {'user-agent' : \
          'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36'}
request = Request(url, headers = headers)
response = urlopen(request)
soup = BeautifulSoup(response, 'html.parser')
soup
titles = soup.select('div.item-title > strong.tit-g.clamp-g > a')
itmes_find_list = [] 
for i, title in enumerate(titles):
    itmes_find_list.append({'no': i, 
                            'title' : title.text,
                            'link' : title.attrs['href']})

import pandas as pd    
pd.DataFrame(itmes_find_list)
# 결과물 없음. 불러올수가 없기 때문에. 브라우저가 접속을 한 것으로 우회를 할 필요가 있음.

,no,title,link
0,0,美·日 국채 팔고 비트코인 샀다…'안전자산' 등극,http://v.daum.net/v/20250528154126316
1,1,[비트코인 2025] 백악관 크립토 차르 “美 정부 비트코인 추가 매입 검토…부채...,http://v.daum.net/v/20250528103907230
2,2,[비즈 나우] 비트코인 2025 컨퍼런스 개막…'전략자산' 선언 코앞,http://v.daum.net/v/20250528075215864
3,3,"블랙록, 자사 비트코인 ETF 보유량 25% 확대…기관 투자 본격화 신호탄",http://v.daum.net/v/20250528151802080
4,4,"비트코인, 트럼프 미디어 비축 소식에도 주춤…1억5100만원대",http://v.daum.net/v/20250528095112897
5,5,"美 상원의원 ""트럼프 대통령, 비트코인법 지지""",http://v.daum.net/v/20250528090342853
6,6,[비트코인 2025] 로빈후드 창업자 “토큰화 증권은 美 ‘자본 패권’ 키우는 수단”,http://v.daum.net/v/20250528110600761
7,7,"트럼프미디어그룹, 25억 달러 규모 자금 조달 통해 비트코인 매입 예고",http://v.daum.net/v/20250528084245020
8,8,"'비트코인 빚투' 스트레티지, 또 샀다…보유량 58만개 돌파",http://v.daum.net/v/20250528042404292
9,9,"숨 고르는 비트코인, 10만8000달러선 '주춤'",http://v.daum.net/v/20250528095921258


### n) yes24 베스트셀러 크롤링 연습

In [127]:
# 방법1
import requests
response = requests.get('https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber=1&pageSize=24')
soup = BeautifulSoup(response.content, 
                    "html.parser")
gd_names = soup.select('a.gd_name')
for i, gd_name  in enumerate(gd_names):
        print(f'{i+1}. {gd_name.text}')

1. 청춘의 독서
2. 엄마의 말 연습
3. 어른의 품격을 채우는 100일 필사 노트
4. 단 한 번의 삶
5. 이로운 보수 의로운 진보
6. 소년이 온다
7. 어른의 행복은 조용하다
8. 초역 부처의 말
9. 모순
10. 빛과 실
11. 쇼펜하우어 인생수업 : 한 번뿐인 삶 이렇게 살아라 (리커버 에디션)
12. 듀얼 브레인
13. 괴수 8호 15 트리플특장판
14. 길고양이와 늑대 5 더블특전판
15. 혼모노
16. 위버멘쉬
17. 결국 국민이 합니다
18. 급류
19. 에그박사 15
20. 첫 여름, 완주
21. 친구가 상처 줄 때 똑똑하게 나를 지키는 법
22. 친구 사이에도 예의가 필요해
23. 서랍에 저녁을 넣어 두었다
24. 워런 버핏 웨이


### 5) 멜론 차트 크롤링
- 꼭 user-agent를 사용해야 하는 경우 
- https://www.melon.com/chart/index.htm

In [264]:
# 방법 1
import requests
from bs4 import BeautifulSoup
url = "https://www.melon.com/chart/"
melon_chart = requests.get(url)
melon_chart  # 406 코드는 막혀있다는 뜻. 1번 방법은 할 수 없다

<Response [406]>

In [278]:
# 방법 2
from urllib.request import urlopen, Request
url = "https://www.melon.com/chart/"
headers = {'user-agent' :
         'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36'}
request = Request(url, headers=headers)
melon_chart = urlopen(request)

#방법 1
melon_chart = requests.get(url, headers=headers)
soup = BeautifulSoup(melon_chart.text, 'html.parser')
soup

<!DOCTYPE html>

<html lang="ko">
<head>
<meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
<meta content="IE=edge,chrome=1" http-equiv="X-UA-Compatible"/>
<title>멜론차트&gt;TOP100&gt;멜론</title>
<meta content="음악서비스, 멜론차트, 멜론TOP100, 최신음악, 인기가요, 뮤직비디오, 앨범, 플레이어, 스트리밍, 다운로드, 아티스트플러스, 아티스트채널" name="keywords"/>
<meta content="No.1 뮤직플랫폼 멜론! 최신 트렌드부터 나를 아는 똑똑한 음악추천까지!" name="description"/>
<meta content="ee85ff6db1fa8f2226bcb671ecb2bcdbcffb6f8b" name="naver-site-verification"/>
<meta content="q4tbTQhmxa4La3OdNLsNOCxrJ_WV6lUlBFrFW4-HqQc" name="google-site-verification"/>
<meta content="4022717807957185" property="fb:app_id"/>
<meta content="Melon" property="og:title"/>
<meta content="https://cdnimg.melon.co.kr/resource/image/web/common/logo_melon142x99.png" property="og:image"/>
<meta content="음악이 필요한 순간, 멜론" property="og:description"/>
<meta content="http://www.melon.com/chart/" property="og:url"/>
<meta content="website" property="og:type"/>
<meta content="멜론" property="og:s

In [304]:
# 멜론 순위, 곡명, 가수명, 좋아요수
# 1위 | 너에게 닿기를 | 10CM | ♥ 85,756
# 곡명 = div.wrap_song_info > div.ellipsis.rank01
# 가수명 = div.wrap_song_info > div.ellipsis.rank02 > span
# 좋아요수 = div.wrap > button > span.cnt
titles = soup.select('div.wrap_song_info > div.ellipsis.rank01')
title = [t.text.strip() for t in titles]
# len(title), title
artists = soup.select('div.wrap_song_info > div.ellipsis.rank02 > span')
artist = [a.text for a in artists]
chart = []
for i, (t, a) in enumerate(zip(title, artist)):
    chart.append({'Rank' : str(i+1)+'위', 
                  'Title': t, 
                  'Artist' : a})

chart_df = pd.DataFrame(chart)
chart_df

,Rank,Title,Artist
0,1위,너에게 닿기를,10CM
1,2위,Never Ending Story,아이유
2,3위,Drowning,WOODZ
3,4위,like JENNIE,제니 (JENNIE)
4,5위,모르시나요(PROD.로코베리),조째즈
...,...,...,...
95,96위,Sticky,KISS OF LIFE
96,97위,Small girl (feat. 도경수(D.O.)),이영지
97,98위,Dash,PLAVE
98,99위,사막에서 꽃을 피우듯,우디 (Woody)
